In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
#in this code let's use the AI we've created to make a battleship game
#Jeremy Kedziora
#4 Nov 2024
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@
#import useful libraries
#@@@@@@@@@@@@@@@@@@@@@@@
import numpy as np
import copy
import seaborn as sb

from tqdm import tqdm
from IPython.display import clear_output
import time
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import pickle


In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
#A class specifically for the reward function to make it easy to swap in and out
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
class Reward:
    
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def __init__(self, config):
        '''Set up the constructor
            Takes -- config, a dictionary specifying the track dimensions and initial state
        '''
        self.config = config    #save the config file
        with open('all_boards.pkl', 'rb') as file:    #open the pickle of all boards
            self.all_boards = pickle.load(file)    #pull into memory
        self.reset_consistent_boards()
        self.reward_map = self.compute_posterior(board_state = self.config['initial_state'])    #update the posterior to that state

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def reset_consistent_boards(self):
        '''A function to reset the boards -- this would need to be called at the end of an episode'''
        self.consistent_boards = copy.deepcopy(self.all_boards['boards'])    #reset the boards
    
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def compute_posterior(self,board_state):
        '''A function to update the set of boards that is consistent with what is currently observed'''
        new_consistent = {}    #init a dictionary to hold the updated boards
        board_mask = (board_state == 0)*1    #find the untried locations
        posterior_distribution = np.zeros([self.config['n'],self.config['n']])    #set up a grid to track cell counts from consistent boards
        for k in self.consistent_boards.keys():    #loop over all currently consistent boards
            c1 = np.sum((self.consistent_boards[k] - board_state) < 0)    #check if the candidate board is empty where the obs board has a hit
            c2 = np.sum((board_state - self.consistent_boards[k]) < -1)    #check if the candidate board is filled where the obs board has a miss
            if c1 + c2 == 0:    #if neither condition holds...
                posterior_distribution = posterior_distribution + self.consistent_boards[k]    #add the information to the posterior
                new_consistent[k] = self.consistent_boards[k]    #add the board if is consistent with the current pattern of hits and misses
        self.consistent_boards = new_consistent
        posterior_distribution = posterior_distribution * board_mask    #zero out locations that have been tried already
        posterior_distribution = posterior_distribution/np.max(posterior_distribution)
        return posterior_distribution

    # def reward(self, s_prime, s):
    #     reward = 0
    #     #action resulted in a hit
    #     if(np.sum(s_prime) > np.sum(s)):
    #         self.hit_streak += 1
    #         reward += (5 * self.hit_streak)         
    #     #action resulted in a miss
    #     elif(np.sum(s_prime) < np.sum(s)):
    #         self.hit_streak = 0
    #         reward -= 1
    #     #penalize repeat action
    #     else:
    #         reward -= 3
        
    #     open_cells = np.sum(s_prime == 0)
    #     # if we are done add more reward
    #     if np.sum(s>0) == sum(self.config['ships']): 
    #         reward += (2 * open_cells)
            
    #     return reward  

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def reward_hit(self,s_pr,a,s):
        '''This reward function looks to see if there is a new hit
            Takes:
                s_pr -- the new state
                s -- the old state
            Returns:
                a scalar reward
        '''
        r = np.sum(s_pr>0) - np.sum(s>0)    #see if there is a new hit
        if np.sum(s_pr == s) == self.config['n']**2:    #if the agent guessed an already tried spot...
            r = -1    #punishment!
        return r

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def reward(self,s_pr,a,s):
        self.reward_map = self.compute_posterior(s)
        # r = self.reward_map[a] + self.reward_hit(s_pr,a,s)
        r = self.reward_hit(s_pr,a,s)
        return r
    

In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
#two classes for these two environments
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
class Battleship:
    '''A class to manage the board'''
    
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def __init__(self, config):
        '''Set up the constructor
            Takes -- config, a dictionary specifying the track dimensions and initial state
        '''
        self.config = config
        self.hit_streak = 0
        self.true_state = np.zeros([config['n'],config['n']],dtype = int)
        self.state = np.zeros([config['n'],config['n']],dtype = int)    #TODO -- consider how to change this
        self.old_state = copy.deepcopy(self.state)    #init a copy of the old state
        self.reward = Reward(config = {'initial_state':self.state,'n':self.config['n']})

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def position_maker(self,ship):
        '''A function to choose a position for a ship
            Takes:
                self -- class instance info above
                ship -- int, the index in the list of ship lengths
            Returns:
                an int, int pair, the coordinates of the first cell the ship occupies
        '''
        coord_1 = np.random.choice(list(range(self.config['n'] - self.config['ships'][ship] + 1)))    #choose first coordinate, guaranteed to have ship on board
        coord_2 = np.random.choice(list(range(self.config['n'])))    #choose second coordinate
        return coord_1,coord_2

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def check_empty(self,coord_1,coord_2,vert,ship):
        '''A function to make sure the proposed location is currently empty
            Takes:
                self -- class instance info above
                coord_1 -- int, the first coordinate, constrained to ensure the ship lies on the board
                coord_2 -- int, the second coordinate
                vert -- boolean, whether to orient the ship vertically or horizontally
                ship -- int, the index in the list of ship lengths
            Returns:
                a boolean, indicating if the placement is legal
        '''
        if vert:    #if the ship is oriented vertically...
            if sum(self.true_state[coord_1 + list(range(self.config['ships'][ship])),coord_2]) > 0:    #and if there is another ship lying across its path...
                empty = False    #set the empty indicator to false
            else:    #otherwise if the proposed location is empty...
                empty = True    #set the empty indicator to true
        else:    #if the ship is oriented horizontally...
            if sum(self.true_state[coord_2,coord_1 + list(range(self.config['ships'][ship]))]) > 0:    #and if there is anther ship lying across its path...
                empty = False    #set the empty indicator to false
            else:    #otherwise if the proposed location is empty...
                empty = True    #set the empty indicator to true
        return empty

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def place_ship(self,ship):
        '''A function to put a single ship on the board
            Takes:
                self -- class instance info above
                ship -- int, the index in the list of ship lengths
        '''
        empty = False    #init the stopping criterion
        while not empty:    #while we should keep going...
            coord_1,coord_2 = self.position_maker(ship)    #sample a position for the ship
            vert = np.random.uniform() < 0.5    #choose whether ship is horizontal or vertical
            empty = self.check_empty(coord_1,coord_2,vert,ship)    #check if legal
        if vert:    #if the ship is vertical...
            self.true_state[coord_1 + list(range(self.config['ships'][ship])),coord_2] = 1    #place on board
        else:    #if the ship is horizontal...
            self.true_state[coord_2,coord_1 + list(range(self.config['ships'][ship]))] = 1    #place on board
        vert = np.random.uniform() < 0.5    #choose whether ship is horizontal or vertical

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def make_board(self):
        '''A function to put all ships on the board
            Takes:
                self -- class instance info above
        '''
        self.true_state = np.zeros([self.config['n'],self.config['n']],dtype = int)
        for j in range(len(self.config['ships'])):    #loop over ship lengths
            self.place_ship(j)    #put on board'

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def reset(self):
        '''A function to reset the board
            Takes:
                self -- class instance info above
        '''
        self.hit_streak = 0
        self.true_state = np.zeros([self.config['n'],self.config['n']],dtype = int)    #reset the hidden state
        self.make_board()    #place ships again
        self.state = np.zeros([self.config['n'],self.config['n']],dtype = int)    #reset the observed state
        self.old_state = copy.deepcopy(self.state)
        
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def check_win(self):
        '''A function to check whether the agent has won
            Takes:
                self -- class instance info above
            Returns:
                a boolean, indicating if the agent sank all the ships
        '''
        if np.sum(self.state>0) == sum(self.config['ships']):    #if the number of hits is equal to the total length of ships...
            return True    #return true -- the agent won!
        else:    #otherwise...
            return False    #return false -- the game is still going
        
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def step(self,a):
        '''A function to update the state given the true enemy fleet locations'''
        if self.true_state[a['row'],a['col']] > 0:    #if there is a ship at the chosen location...
            self.state[a['row'],a['col']] = 1    #record a hit on the observed state as a 1
        else:    #if there is no ship
            self.state[a['row'],a['col']] = -1    #record a miss on the observed state as a -1
        done = self.check_win()    #check to see if the agent has won
        if done:
            r = 10#max(10*np.sum(self.state == 0),10)
        else:
            r = self.reward.reward(self.state,(a['row'],a['col']),self.old_state)
        self.old_state = copy.deepcopy(self.state)
        return{'state':self.state,'reward':r,'done':done}
        

In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
#a CNN architecture
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        # Input: (batch_size, 1, 6, 6)
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)  # -> (32, 6, 6)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # -> (64, 6, 6)
        self.conv3 = nn.Conv2d(64, 1, kernel_size=1)  # -> (1, 6, 6)

    def forward(self, x):
        x = F.relu(self.conv1(x))  # (batch_size, 32, 6, 6)
        x = F.relu(self.conv2(x))  # (batch_size, 64, 6, 6)
        q_map = self.conv3(x)      # (batch_size, 1, 6, 6)
        return q_map.squeeze(1)    # -> (batch_size, 6, 6)


In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
class Agent:
    '''A class to manage the chase'''
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def __init__(self, config):
        '''Set up the constructor
            Takes -- config, a dictionary specifying the track dimensions and initial state
        '''
        self.config = config    #save the config file
        self.performance_memory = {}
        self.A_s = self.config['A']    #save the action set
        self.Q = self.config['Q_model']    #init the deep learning model
        if self.config['warm_start']:    #if we have loaded a pretrained agent...
            self.Q = self.config['warm_start']['agent']    #use this for the Q-value model
            self.performance_memory = self.config['warm_start']['performance']    #and load in the performance data
        self.Q_target = copy.deepcopy(self.Q)    #copy the target approximator
        self.D = []    #init the replay buffer
        self.criterion = torch.nn.MSELoss()
        self.optimizer = optim.Adam(self.Q.parameters(), lr=self.config['alpha'])
        
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def update_Q_target(self):
        '''A function to update the target approximator
        '''
        self.Q_target = copy.deepcopy(self.Q)    #copy the target approximator
        
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def purge_replay_buffer(self):
        '''A function to keep the replay buffer within memory limits
        '''
        if len(self.D) > self.config['M']:    #if the replay buffer has exceeded memory size...
            self.D.pop(0)    #remove the oldest element

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def forward_pass(self,s_t,main = True,mask_illegal = True,grad = True):
        '''A function to compute the forward pass
        '''
        s_t_tensor = torch.tensor(s_t, dtype=torch.float32)    #make into tensor
        s_t_tensor = s_t_tensor.unsqueeze(0).unsqueeze(0)    #put into proper format
        
        if grad:   #if you want to retain gradient infomation...
            if main:    #if you want to use the main function approximator...
                q_map = self.Q(s_t_tensor).squeeze(0)    #generate a map of q-values using the forward pass on the main function approximator
            else:    #if you want to use the target approximator
                q_map = self.Q_target(s_t_tensor).squeeze(0)    #generate a map of q-values using the forward pass on the target approximator            
        if not grad:   #if you don't want to retain grad information...
            with torch.no_grad():    #disables gradient calculation for the following forward pass
                if main:    #if you want to use the main function approximator...
                    q_map = self.Q(s_t_tensor).squeeze(0)    #generate a map of q-values using the forward pass on the main function approximator
                else:    #if you want to use the target approximator
                    q_map = self.Q_target(s_t_tensor).squeeze(0)    #generate a map of q-values using the forward pass on the target approximator
        legal_mask = (s_t_tensor == 0)    #create a map that shows the open spaces
        legal_positions = legal_mask.nonzero(as_tuple=False)    #pull out the pairs corresponding to the open spaces
        if mask_illegal:    #if you want to mask out already taken actions...
            q_map[~legal_mask[0][0]] = float('-inf')    #Set illegal positions to -inf so argmax ignores them
        return q_map
    
    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@    
    def pi(self, s_t, epsilon,print_map = False):
        '''A function to choose actions using Q-values FOR THE ES architecture
            Takes:
                s_t -- 6 x6 battleship state grid
                epsilon -- the probability of choosing a random action
        '''
        if np.random.uniform() < epsilon:    #with small probability...
            return tuple(self.A_s[np.random.choice(a=range(len(agent.A_s)))])    #return a random action
        else:    #with large probability...
            q_map = self.forward_pass(s_t,main = True,mask_illegal = True,grad = False)    #generate Q-values from the main function approximator
            return tuple(self.A_s[torch.argmax(q_map.view(-1)).item()])

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def make_batch(self):
        '''A function to make a batch for updating Q
        '''
        return np.random.choice(a = self.D,size = self.config['B'])

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def update_Q_value(self,batch):
        Qs = []    #init a list to hold the Q-values for the states and actions
        y = []    #init a list to hold the target
        for d in batch:    #loop over data in batch
            y_d = d['r']    #set the target to the reward value
            if not d['done']:    #if s' did not end the episode...
                q_map = self.forward_pass(d["s'"],main = False,mask_illegal = False,grad = False)    #generate Q-values with the target function approximator
                y_d += self.config['gamma']*torch.max(q_map)    #take the max, discount, and add the reward
            y.append(y_d)    #save the target
            Q_d = self.forward_pass(d["s"],main = True,mask_illegal = False,grad = True)    #generate Q-values with the main function approximator
            Qs.append(Q_d[d['a']])    #pull out the Q-value for the actual action
        Qs = torch.stack(Qs).unsqueeze(1)    #put the Q-values in the proper form
        y = torch.tensor(y).to(torch.float32).unsqueeze(1)    #make target info into tensor
        loss = self.criterion(Qs,y)    #compare Q-values to targets to create the loss
        
        # #do the backward pass
        self.optimizer.zero_grad()    #zero out the gradients    
        loss.backward()    #compute gradients
        self.optimizer.step()    #perform a single optimzation step

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@            
    def rolling_average(self,data):
        data_summary = []    #init a list to hold rolling average
        for j in range(len(data)):    #loop over data
            if j % 100 == 0 and j > 0:    #if we hit the right index...
                data_summary.append(np.mean(data[j-100:j]))    #compute the mean
        return data_summary

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def make_plot(self,data_summary,ylims,x_label,y_label,title,file_name):
        fig, ax = plt.subplots()    #init the plot
        sb.lineplot({x_label:np.array(range(len(data_summary)))*100 + 100
                     ,y_label:data_summary}
                    ,x = x_label
                    ,y = y_label
                    ,ax = ax)    #make the lineplot
        plt.ylim(ylims[0],ylims[1])    #set the axes
        ax.set_title(title)    #set the plot title
        fig.savefig(file_name, dpi=400)    #save to file
        plt.close(fig)    #Close the figure

    #@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
    def save_agent(self):
        with open(self.config['file_name'], 'wb') as f:
            pickle.dump({'agent':self.Q,'performance':self.performance_memory},f)

        self.make_plot(data_summary = self.rolling_average(self.performance_memory['n_turns'])
          ,ylims = (0,36)
          ,x_label = 'Episode'
          ,y_label = 'Average Turns'
          ,title = 'Average Turns to Finish Battleship'
          ,file_name = 'n_turn_summary.png')

        self.make_plot(data_summary = self.rolling_average(self.performance_memory['return'])
          ,ylims = (0,15)
          ,x_label = 'Episode'
          ,y_label = 'Average Return'
          ,title = 'Average Return during Training'
          ,file_name = 'return_summary.png')


In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
#Set up the classes -- the environment and the agent
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
env_config = {'n':6,'ships':[2,3,4],'boards':None}    #set up the environmental config

#set up the agent; actions, warm start, hyperparameters
A = []    #init the actions
for row in range(6):    #loop over vertical velo changes
    for col in range(6):    #loop over horizonal velo changes
        A.append([row,col])    #record

#the warm started agent
with open('trained_agent.pkl', 'rb') as file:    #open the pickle of all boards
    trained_agent_data = pickle.load(file)    #pull into memory

#the hyperparameters
agent_config = {'gamma':0.9    #the discount factor
                ,'epsilon':0.1    #the epsilon-greedy parameter
                ,'alpha':0.001    #the learning rate
                ,'Q_model':CNN()    #the model architecture
                ,'A':A    #the action set
                ,'n':6    #the size of the board
                ,'M':100000    #set the memory size
                ,'B':20    #set the batch size
                ,'C':500    #when to update the target approximator
                ,'n_steps_for_Q_update':4    #the number of steps to use to update
                ,'warm_start':trained_agent_data
                ,'save_every':500    #write out the agent and its performance every this many steps
                ,'file_name':'trained_agent.pkl'
                # ,'epsilon_burnin':10000
                # ,'omega':0.5
                # ,'prioritized_replay':True
                # ,'double_Q':True
                # ,'td_n':5
                # ,'checkpoints':2000
               }   #set up the agent config

#init the environment and agent
board = Battleship(env_config)    #set up the environment for testing agent performance
agent = Agent(agent_config)    #set up the agent


In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@
#Run the DQN training loop
#@@@@@@@@@@@@@@@@@@@@@@@@@
N_eps = 10000   #specify the number of episodes
agent.performance_memory['n_turns'] = []
agent.performance_memory['return'] = []
total_turns = 0
for epi in tqdm(range(N_eps)):    #loop over episodes
    board.make_board()    #set up the board
    done = False    #set the stopping condition
    turn = 0    #init the episode turn count
    episode_return = 0.0
    while not done:#for _ in range(6):    #while the episode goes on...
        total_turns += 1    #add to the total turn count
        a = agent.pi(s_t = board.state,epsilon = 0.1,print_map = False)    #generate an action
        
        data_t = {}    #init the tuple for this turn
        data_t['s'] = copy.deepcopy(board.state)    #save the old state
        data_t['a'] = a    #save the action
        next_step = board.step({'row':a[0],'col':a[1]})    #evolve the environment
        episode_return += (agent.config['gamma']**turn)*next_step['reward']
        data_t['r'] = next_step['reward']    #save the reward
        data_t["s'"] = copy.deepcopy(next_step['state'])    #save the new state
        
        agent.D.append(data_t)    #add this tuple to the replay buffer
        agent.purge_replay_buffer()    #remove the oldest tuple if appropriate
        
        done = next_step['done']    #record whether we've stopped
        data_t['done'] = done
        
        if total_turns % agent.config['n_steps_for_Q_update'] == 0:
            batch = agent.make_batch()
            agent.update_Q_value(batch)
                  
        if total_turns % agent.config['C'] == 0:    #if we've gone another C turns...
            agent.update_Q_target()    #update the target approximator
        turn += 1    #add to the episode turn count
        
    board.reset()    #reset the board
    board.reward.reset_consistent_boards()
    agent.performance_memory['n_turns'].append(turn)    #record the number of turns
    agent.performance_memory['return'].append(episode_return)    #record the number of turns

    if total_turns % agent.config['save_every'] == 0:
        agent.save_agent()


In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
#Run the trained DQN agent at inference
#@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@
N_eps = 10   #specify the number of episodes
for epi in tqdm(range(N_eps)):    #loop over episodes
    board.make_board()    #set up the board
    done = False    #set the stopping condition
    turn = 0    #init the episode turn count
    episode_return = 0.0
    while not done:#for _ in range(6):    #while the episode goes on...
        a = agent.pi(s_t = board.state,epsilon = 0.1,print_map = False)    #generate an action
        
        next_step = board.step({'row':a[0],'col':a[1]})    #evolve the environment
        episode_return += (agent.config['gamma']**turn)*next_step['reward']
                
        done = next_step['done']    #record whether we've stopped
        turn += 1    #add to the episode turn count
        
    board.reset()    #reset the board
    board.reward.reset_consistent_boards()
    print('The agent took',turn,'turns to finish with return',episode_return)
    

In [ ]:
#@@@@@@@@@@@@@@@@@@@@@@@@@@
#Play a game against the AI
#@@@@@@@@@@@@@@@@@@@@@@@@@@
env_config = {'n':6,'ships':[2,3,4],'boards':None}    #set up the environmental config
player_board = Battleship(env_config)    #set up the environment for testing agent performance
AI_board = Battleship(env_config)    #set up the environment for testing agent performance
player_board.make_board()    #set up the player board
AI_board.make_board()    #set up the AI board

player_win = False
ai_win = False
while not player_win and not ai_win:
    print('Here is your board:')
    for row_player,row_hits_on_player,row_ai in zip(player_board.true_state,player_board.state,AI_board.state):
        row_string = '|'
        for space,hit in zip(row_player,row_hits_on_player):
            if space == 0 and hit == 0:
                row_string = row_string + '~|'
            if space == 1 and hit == 0:
                row_string = row_string + 'S|'
            if space == 0 and hit == -1:
                row_string = row_string + 'M|'
            if space == 1 and hit == 1:
                row_string = row_string + 'H|'            
        row_string_ai = '|'
        for space in row_ai:
            if space == 0:
                row_string_ai = row_string_ai + '~|'
            if space == -1:
                row_string_ai = row_string_ai + 'M|'
            if space == 1:
                row_string_ai = row_string_ai + 'H|'
        print(row_string,'   ',row_string_ai)
    print('')
    a_player = input("Enter where would you like to strike (as row,column).")
    next_step = AI_board.step({'row':int(a_player[0])-1,'col':int(a_player[2])-1})
    player_win = next_step['done']
    if player_win:
        print('You won!')
        break
    a_ai = agent.pi(s_t = player_board.state,epsilon = 0.1,print_map = False)    #generate an action
    next_step = player_board.step({'row':a_ai[0],'col':a_ai[1]})    #evolve the environment
    ai_win = next_step['done']
    if ai_win:
        print('You lost!')
    # clear_output(wait=True) # Setting wait=True provides a smoother transition for dynamic updates
    